Okay I can kind of see it now...

    * THE SUPREME ENTRY POINT: ./aic/aic_model/aic_model/aic_model.py
            - This contains the main function, and it has a series of scalpers
              from line 62-122 which extracts the various input parameters
              given from the entry command.
`ros2 run aic_model aic_model --ros-args -p policy:=aic_example_policies.ros.CheatCode`              
            - aic_model.py's AicModel(LifecycleNode) class then scalps the
              class definition of `aic_example_policies.ros.CheatCode` in line
              67 `inspect.getmembers(policy_module, inspect.isclass)`.
            - The `CheatCode.py`'s class is defined as CheatCode(policy) --
              which inherits from ./aic/aic_model/aic_model/policy.py
              ... and hence the full circle is reached and everything ties
              together nicely...

            - Which all the client-side (our-side)'s configs are passed in
              from `class CheatCode(polict)`... which this is where we 
              implement the list of 10 policy models, and other fun stuff
              too...

---

---
TOP 10 FILES FOR THE AIC Competition -- Deep Walkthrough

---
FILE 1: `policy.py` - The Contract You Must Fulfill

PATH: `aic_model/aic_model/policy.py` (146 lines)
IMPORTANCE: This is THE file. Everything you build inherits from this.


KEY LINES EXPLAINED
    Lines 36-68 - THE THREE CALLBACKS (your API to the world)

    `GetObservationCallback = Callable[[], Observation]     # Line 36`
    This is a function type. When you call `get_observation()` inside your
    policy, it returns the latest `Observation` message -- all 3 cameras, joint
    states, wrench data, controller state. Delivered at 20 Hz. 

```Python
class MoveRobotCallback(Protocol):
    def __call__(
        self,
        motion_update: MotionUpdate = None,                 # Cartesian control
        joint_motion_update: JointMotionUpdate = None,      # Joint control
    ) -> None: ...
```
   This is how you command the robot. You pass EITHER a `MotionUpdate` 
   (Cartesian pose/velocity) OR a `JointMotionUpdate` (joint angles). Never
   both at once. The system auto-switches control modes for you.


```Python
SendFeedbackCallback = Callable[[str], None]                # Line 68
```
   Sends a progress string back to the engine. Use it for logging/debugging.


---
LINES 71-88 -- The Policy Base Class
```Python
class Policy(ABC):
    def __init__(self, parent_node):
        self._parent_node = parent_node         # Stores the ROS node
```
   `parent_node` is the `AicModel` lifecycle node. Through it you access the
   ROS clock, logger, and TF buffer. This is critical -- 
   `self._parent_node._tf_buffer` is how CheatCode looks up transforms.

```Python
    def time_now(self):
        return self.get_clock().now()

    def sleep_for(self, duration_sec: float) -> None:
        self.get_clock().sleep_for(Duration(seconds=duration_sec))
```
   These use SIM TIME, not wall time. The sim clock can run faster or slower
   than real time. Always use these instead of `time.sleep()` for anything
   physics-coupled. 


---
LINES 90-134 -- `set_pose_target()` -- The Convenience Helper
```Python
    def set_pose_target(
        self, 
        move_robot: MoveRobotCallback, 
        pose: Pose, 
        frame_id, 
        str = "base_link"
    ) -> None:
```
   This wraps up a `MotionUpdate` message with sensible defaults. The key 
   parameters it sets:

```Python
        motion_update = MotionUpdate(
            target_stiffness=np.diag([90.0, 90.0, 90.0, 50.0, 50.0, 50.0]).flatten(),  # Line 120
            target_damping=np.diag([50.0, 50.0, 50.0, 20.0, 20.0, 20.0]).flatten(),    # Line 121
        )
```
   - Stiffness [90, 90, 90, 50, 50, 50]: How strongly the arm tracks the target
     pose. First 3 = translational (N/m), last 3 = rotational (Nm/rad). Higher
     = stiffer tracking, more jerk. 
   - Damping [50, 50, 50, 20, 20, 20]: Velocity-proportional resistance. 
     Higher = smoother but slower. 


```Python
        wrench_feedback_gains_at_tip=[0.5, 0.5, 0.5, 0.0, 0.0, 0.0], # Line 126
```
   Force feedback gains. [0.5, 0.5, 0.5] means 50% force feedback on translation
   (the arm yields to external forces). [0, 0, 0] on rotation means no 
   rotational compliance. Range is [0, 0.95].


```Python
              trajectory_generation_mode=TrajectoryGenerationMode(
                  mode=TrajectoryGenerationMode.MODE_POSITION,               # Line 128
              ),
```
   `MODE_POSITION` means the pose field is used. `MODE_VELOCITY` means the
   velocity field is used instead. 


---
LINES 136-145 -- The Abstract Method You Implement
```Python
        @abstractmethod
        def insert_cable(
            self,
            task: Task,                                 # What to insert where
            get_observation: GetObservationCallback,    # See the world
            move_robot: MoveRobotCallback,              # Command the arm
            send_feedback: SendFeedbackCallback,        # Report progress
        )
```
   




---
1. THE `task` (TASK MESSAGE)
   The `task` object is a data container (a ROS message) that tells your AI
   exactly what its objective is for the current rail. It essentially defines 
   the MISSION PARAMETERS.

   WHAT IT LOOKS LIKE IN YOUR CODE:
      When your `insert_cable` method is called, you can access the fields like
      this:
```Python
def insert_cable(self, task, get_observation, move_robot, send_feedback):
    # Logging the task details to your console
    self.get_logger.info(f"Objective ID: {task.id}")
    self.get_logger.info(f"Targeting module: {task.target_module_name}")
    self.get_logger.info(f"Inserting: {task.plug_type} into {task.port_name}")

    # Logic: If the task is an SFP module, we might use a different vision
    # model.
    if task.plug_type == "sfp":
        target_color = "silver"
    elif task.plug_type == "sc":
        target_color = "blue"
```

   BREAKDOWN OF THE "STUFF":
   * `task.id`: A unique string (e.g., `"trail_01_insertion"`). Useful for 
     logging data if you're recording a dataset.
   * `task.cable_name`: The specific physical instance (e.g, `"cable_0"`). 
     Important because different cables might have different physical properties
     in the sim. 
   * `task.plug_type`: The kind of connector (e.g., `"sfp"`, `"lc"`, or `"sc"`).
     This tells you which CAD model/visual features to look for.
   * `task.port_name`: The target destination (e.g, `"sfp_port_0"`)
   * `task.time_limit`: An integer (seconds). If you haven't returned `True` by
     this time, the `aic_engine` will kill your node and give you a score of 0
     for that trail. 



---
2. THE `move_robot` (MoveRobotCallback)

```Python
        motion_update = MotionUpdate(
            header=Header(
                frame_id=frame_id,
                stamp=self._parent_node.get_clock().now().to_msg(),
            ),
            pose=pose,
            target_stiffness=np.diag([90.0, 90.0, 90.0, 50.0, 50.0, 50.0]).flatten(),
            target_damping=np.diag([50.0, 50.0, 50.0, 20.0, 20.0, 20.0]).flatten(),
            feedforward_wrench_at_tip=Wrench(
                force=Vector3(x=0.0, y=0.0, z=0.0),
                torque=Vector3(x=0.0, y=0.0, z=0.0),
            ),
            wrench_feedback_gains_at_tip=[0.5, 0.5, 0.5, 0.0, 0.0, 0.0],
            trajectory_generation_mode=TrajectoryGenerationMode(
                mode=TrajectoryGenerationMode.MODE_POSITION,
            ),
        )
```

```Python
from aic_control_interfaces.msg import MotionUpdate, TrajectoryGenerationMode
from geometry_msgs.msg import Pose, Point, Quaternion

# 1. Create the data package (MotionUpdate)
cmd = MotionUpdate()

# 2. Set the goal position (e.g., 5cm below current hover point)
cmd.pose.position = Point(x=-0.4, y=0.45, z=0.20) 
cmd.pose.orientation = Quaternion(x=1.0, y=0.0, z=0.0, w=0.0)

# 3. Set the "Feel" (Impedance & Compliance)
# High stiffness in Z (push hard), low stiffness in X/Y (let it wiggle)
cmd.target_stiffness = [50.0, 50.0, 200.0, 30.0, 30.0, 30.0] 
# Give 50% translation compliance so it yields if it hits the port edge
cmd.wrench_feedback_gains_at_tip = [0.5, 0.5, 0.5, 0.0, 0.0, 0.0]

# 4. Choose the mode
cmd.trajectory_generation_mode.mode = TrajectoryGenerationMode.MODE_POSITION

# 5. EXECUTE: This is the actual MoveRobotCallback in action
move_robot(motion_update=cmd)
```

   BREAKDOWN OF THE "STUFF":
   * `motion_update`: This is the argument name expected by the callback.
   * `cmd.pose`: Where the robot wants to be.
   * `cmd.target_stiffness`: How "stubborn" the robot is about getting to that
     pose. High values make it a rigid industrail robot; low values make it a 
     soft "collaborative" robot.
   * `cmd.wrench_feedback_gains_at_tip`: As explained before, this is the ACTIVE
     FORCE RESPONSE. It uses the Axia80 sensor data to actively move the arm in
     the direction of felt forces to minimise the stress on the cable.   


---
FILE 2: `aic_model.py` -- THE LIFECYCLE NODE (Your Runtime)
   
   PATH: `aic_model/aic_model/aic_model.py` (335 lines)
   IMPORTANCE: Understand this to know what happens around your code. 


KEY LINES EXPLAINED   

LINES 53-80 -- Dynamic Policy Loading
```Python
class AicModel(LifecycleNode):
    def __init__(self):
        super().__init__("aic_model")
        self.declare_parameter("policy", "WaveArm") 
        policy_module_name = (
          self.get_parameter("policy").get_parameter_value().string_value
        )
        policy_module = importlib.import_module(policy_module_name)
```

   Your policy is loaded dynamically via a string name. When you run 
   `ros2 run aic_model aic_model --ros-args -p policy:=my_package.MyPolicy`, it
   imports `my_package.MyPolicy` and looks for a class named `MyPolicy` inside.


LINES 81-84 -- TF Buffer Setup   




























... You can think of the DYNAMIC POLICY LOADING in `AicModel` as the "Main
Selector." However, there's a slight distinction in how you'd execute that
switching. Usually, you wouldn't restart the whole ROS ndoe to switch between
`.pt` files; instead, you would implement one "Master Policy" class that 
contains the logic to load and swap those 10 different models internally based 
on the `task` message you receive. When `insert_cable` is called, your code 
looks at `task.plug_type`, and if it sees "SFP," it routes the camera frames
through `model_1.pt`; if it sees "LC," it switches the pipeline to `model_5.pt`.

This setup is incredibly powerful for your...


---
   In the standard toolkit provided by Intrinsic, there isn't a pre-build "array
   of paths" parameter. You have to build that "Entry Point" logic yourself.

   The ARCHITECTURAL ENTRY POINT for this is a two-step process: you declare a
   custom parameter in the ROS 2 node, and then you handle that data inside your
   Policy's constructor.


1. THE "CONFIG" ENTRY POINT (COMMAND LINE)
   When you launch your node, you pass strings. To pass 10 different paths, you
   would typically point to a YAML configuration file that contains that array,
   or pass a single directory path that contains all your `.pt` files.

   In your terminal, it looks like this:
```bash
ros2 run aic_model aic_model --ros-args -p policy:=my_package.MasterPolicy -p 
    models_path:="/path/to/my/models"
```
                    <-- ... i think this is false... as we would need to modify
                        `aic_model.py` in this case :\


2. THE "CODE" ENTRY POINT (`__init__`)
   Inside your custom `MasterPolicy` class, you use the `parent_node` (which is
   the `AicModel` lifecycle node) to grab those paths and load your models into
   a dictionary during startup. 

   Here is how you would structure that file to manage your 10 different models:
```Python
import torch
import os
from aic_model.policy import Policy

class MasterPolicy(Policy):
    def __init__(self, parent_node):
        super().__init__(parent_node)

        # 1. Access the "Entry Point" parameter from the ROS node
        # We assume you've placed all .pt files in one folder
        model_dir = "/Users/administrator/Black Projects/Project-Automaton/models/aic/"
        
        # 2. Load all 10 models into a "Brain Map"
        # This happens ONCE when the node starts (on_configure)
        self.brains = {
            "sfp": torch.load(os.path.join(model_dir, "sfp_insert_v1.pt")),
            "lc":  torch.load(os.path.join(model_dir, "lc_insert_v4.pt")),
            "sc":  torch.load(os.path.join(model_dir, "sc_insert_final.pt")),
            "navigation": torch.load(os.path.join(model_dir, "move_to_board.pt"))
        }
        self.get_logger().info("Master Brain initialised with 10 models.")

                                            # send_feedback :: SendFeedbackCallback :: () -> Action
    def insert_cable(self, task, get_observation, move_robot, send_feedback):   
        # 3. THE SWITCHER: Pick the right brain based on the `task` message
        #    Use task.plug_type to choose which `.pt` file logic to run
        current_brain = self.brains.get(task.plug_type, self.brains["navigation"])

        send_feedback(f"Switching logic to {task.plug_type} specialised model")

        # Run your inference loop using `current_brain`
        # ... logic here ...
        return true

```







---


   In Python, a callback is a function that you pass as an argument to another
   function. The receiving function can then callback at a later point, often
   as a response to an event or after completing a task. 



---
ROS CLOCK
   The ROS clock is the centralised timekeeping mechanism for the entire robot
   system, and in the AIC competition, it is accessed via `self.time_now()` or
   `self.get_clock()`. Unlike standard system time, the ROS clock follows
   SIMULATION TIME when running in Gazebo or Isaac Sim, meaning it can be paused
   , slowed down, or sped up without breaking the logic of your policy. You must
   use this clock for all timing operations--such as the `self.sleep_for()` 
   method--to ensure that your robot's movements remain synchronised with the 
   physics engine. Using a standard Python `time.sleep()` would cause the robot
   to "desync" because the real-world clock keeps ticking even if the simulation
   pauses to calculate a complex collision.


ROS LOGGER
   The ROS logger is the standard diagnostic tool for tracking your policy's
   internal state and errors, accessed through `self.get_logger()`. It allows
   you to output messages at different severity levels--such as `info`, `warn`,
   or `error`--which are then captured by the ROS 2 framework and can be viewed
   in the terminal or saved to log files for post-trail analysis. During the
   cable insertion task, you can use `send_feedback()` to pass progress strings
   back to the engine, but the logger is your primary tool for deep technical
   debugging, such as printing out current force-torque values or the status of
   your vision model. This is essential for understanding why a trail railed 
   once the robot is running autonomously in a Docker container where you cannot 
   see the live 3D viewport. 


TF BUFFER
   The TF (Transform) buffer is a dedicated data structure that tracks the 3D
   coordinate relationships between every part of the robot and its environment
   over time. In your policy, you access this via `self._parent_node._tf_buffer`
   , which allows you to "look up" where the plug tip is located relative to the
   SFP port. The buffer is constantly updated by a `TransformListener` that 
   listens to the `/tf` and `/tf_static` topics. While the `CheatCode.py`
   reference uses this buffer to find the exact "ground truth" location of ports
   , in the actual evaluation, these environmental frames will be hidden, and
   you will primarily use the buffer to track the robot's own internal kinematic
   chain (e.g., from `base_link` to the `hand_e_gripper`), 





---
   ... difference between these two updates defines how you "speak" to the UR5e
   robot's controller.


1. MotionUpdate (Cartesian Control)
   This moves the robot's END-EFFECTOR (TCP) in 3D space. You are telling the
   robot where the gripper should be relative to the world or the robot's base. 

   * TARGET: Uses `geometry_msgs/Pose` (XYZ coordinates and a Quaternion for 
     rotation).
   * PHYSICS: Employs CARTESIAN IMPEDANCE CONTROL. You define a 6x6 stiffness
     matrix and a 6x6 damping matrix to determine how "soft" or "hard" the arm
     tracks that point.
   * BEST USE CASE: The INSERTION PHASE. You need the plug to stay perfectly
     aligned with the port's XYZ coordinates in the world. 


2. JointMotionUpdate (Joint-Space Control)
   This moves the robot's INDIVIDUAL MOTORS. You are telling the robot what
   angle each of its 6 joints should be. 

   * TARGET: Uses `trajectory_msgs/JointTrajectoryPoint` (a list of 6 radian
     values for the UR5e joints).
   * PHYSICS: Employs JOINT-SPACE IMPEDANCE CONTROL. Instead of a 6x6 matrix, 
     you define simple stiffness and damping values for each specific joint. 
   * BEST USE CASE: The APPROACH or HOME PHASE. When moving the arm from its
     starting position to near the task board, joint control is often smoother
     and avoids mathematical "singularities".  


KEY COMPARISON
   Feature // `MotionUpdate` //  `JointMotionUpdate`
   INPUT TYPE // Pose (Position + Rotation) // Joint Angles (Radians)
   LOGIC // "Put the SFP plug at this XYZ" // "Rotate Joint 3 by 45 degrees"
   IMPEDANCE // 6x6 matrices (row-major) // 6 individual joint values
   AUTO-SWITCHING // Handled by `aic_model.py` // Handled by `aic_model.py`

   
   The `aic_model.py` node is smart enough to see which message you sent and
   automatically call the `ChangeTargetMode` service to switch the controller
   for you.




-- An end-effector, often called End-of-Arm Tooling (EOAT), is the device 
   attached to the end of a robotic arm that allows it to interact with its 
   environment, acting as the robot's "hand". These tools enable robots to 
   perform specific tasks such as gripping, welding, cutting, or sensing. Common
   types include mechanical grippers, vacuum suction cups, and specialised tools
   for manufacturing. 



-- Tool Center Point (TCP) in robotics is the precise, user-defined coordinate
   point at the tip of robot's tool (e.g., welder, gripper, or nozzle) that the
   controller uses for motion planning and path execution. It enables accurate
   navigation by shifting the coordinate system from the robot base to the tool
   tip, allowing for rotation around that point rather than the robot flange.
   Correct TCP calibration is essential for precision, repeatability, and 
   preventing collisions. 


-- Cartesian impedance control is a robot motion strategy that regulates the
   relationship between end-effector motion (position/orientation) and external
   forces in Cartesian space, allowing for compliant, human-safe interaction.
   It simulates a virtual spring-damper system, enabling the robot to adjust to
   unknown environments and, if necessary, be guided by hand. 



   This section is where the high-level logic of your AI meets the low-level
   physics of the robot arm. It defines the INTERACTION BEHAVIOR--how the robot
   reacts when it touches something.

   In cable management, "blind" movemenjt is dangerous because cables are 
   flexible and ports are fragile. These lines allow you to give the robot
   "soft hands".
---
   

1. WRENCH FEEDBACK GAINS (Compliance)

```Python
wrench_feedback_gains_at_tip = [0.5, 0.5, 0.5, 0.0, 0.0, 0.0]
```
   This is essentially your "YIELD FACTOR." It tells the robot how much to "give
   in" when it feels a force.

   * THE ARRAY STRUCTURE: It follows the standard 6-axis format: 
     `[Force_X, Force_Y, Force_Z, Torque_X, Torque_Y, Torque_Z]`.
   * TRANSLATION (0.5, 0.5, 0.5): By seeing these to 0.5, you are making the
     robot COMPLIANT. If the robot is moving toward the port and hits a snag, 
     it will "yield" 50%. It won't just keep pushing like a rigid machine; it
     will feel the resistance and slow down or move slightly to the side. 
   * ROTATION (0.0, 0.0, 0.0): By setting these to 0.0, you are making the
     orientation RIGID. The robot will fight to keep the plug exactly at the
     angle you commanded, even if it feels a twisting force. 

   COMPETITION STRATEGY: During the "approach" phase, you want low gains (high
   rigidity). During the "insertion" phase, you want HIGHER GAINS (more 
   compliance) so the cable can self-align as it enters the hole. 


---
2. TRAJECTORY GENERATION MODE

```Python
trajectory_generation_mode=TrajectoryGenerationMode(
    mode=TrajectoryGenerationMode.MODE_
)
```
   This tells the AIC Controller WHICH FIELD in your `MotionUpdate` message to
   prioritise.
      * `MODE_POSITION`: The robot acts like a standard waypoint follower. It
        looks at the `pose` you sent and calculates the best path to get there.
        It's best for the initial "reach" toward the task board. 
      * `MODE_VELOCITY`: The robot ignores the `pose` and instead follows the
        `velocity` (Twist) you send. This is what you see in the `RunACT.py` 
        neural network example. AI models (like ACT) often prefer outputting
        velocities because it allows for smoother, continuous corrections at
        high frequencies (20 Hz+).

    KEY MODIFICATION: If you are a classical "Move here, then move here" script,
    stay in `POSITION`. If you are training a RL/IL agent that redicts the next
    "nudge," switch to `VELOCITY`.


---
2. THE `insert_cable` ENTRY POINT
   This is the "main loop" of your specific challenge logic. When the 
   `aic_engine` decides a trail has started, it calls this method. 

   * `task : Task`: This tells you exactly what the hardware is. For example,
     `task.cable_name` might be "cable_0" and `task.port_name` might be 
     "sfp_port_1". You use these strings to look up the right visual target. 
   * `get_observation: GetObservationCallback`: Calling this is like the robot
     "opening its eyes." It returns the composite message containing the 3 
     Basler camera feeds and the Force/Torque data.
   * `move_robot: MoveRobotCallback`: This is how you "act." You pass your
     `MotionUpdate` (with the compliance gains discussed above) into this 
     function to make the arm move.
   * `send_feedback: SendFeedbackCallback`: This doesn't move the robot, but 
     it's vital for YOU. You send strings like "Searching for port" or 
     "Alignment achieved." These appear in the evaluation logs and help you
     debug why a model failed a trail.   






---

  For tomorrow (Isaac Sim integration):
  1. Get Isaac Sim running with the UR5e model (you already know how to spawn articulations from
  Tutorial 3)
  2. Port the AIC task board description into Isaac Lab's USD format
  3. Set up the ROS 2 bridge between Isaac Sim and the AIC controller stack
  4. Test with a simple policy (like WaveArm) to verify the full pipeline works


  The AIC repo uses Gazebo as its default simulator, but the architecture is simulator-agnostic through
   ROS 2. The key is getting Isaac Sim to publish the same topics (/observation, camera images, joint
  states, F/T data) and subscribe to the same commands (MotionUpdate, JointMotionUpdate) that the AIC
  controller expects.


---    
3. WHAT DOES `write_root_*` DO?
    You are correct--it writes to the SIMULATION BUFFER.

    In high-peformance simulation, you don't move objects by talking to the "UI"
    or the Scene Tree. That is too slow. Instead:

    1. `write_root_pose_to_sim`: This teleports the obejcts to the new XYZ
       coordinates and rotations you just calculated.    
    2. `write_root_velocity_to_sim`: This sets their speed. In this script, it
       usually      

                    https://gemini.google.com/app/d4324975ae77fb67

hi. thanks. can you now on the other hand explain to me how does run_articulation.py work... and how it differs from run_rigid_object.py? thanks!

                    https://gemini.google.com/app/d4324975ae77fb67

finally! please similarly do such too for run_deformable_object.py

                    https://gemini.google.com/app/d4324975ae77fb67